# Stratified Sampling — ViEcomNER Annotation Pool

Pipeline xây dựng annotation pool 3.300 tiêu đề từ corpus sạch.  
Mỗi section được tổ chức theo thứ tự phụ thuộc logic: phân loại đặc trưng → phân bổ hạn ngạch → bổ chính tầng thiểu số → phân chia tập con → xuất file.


## 1. Imports và cấu hình

Nạp thư viện và tải từ điển tiếng Anh (NLTK) dùng cho bộ phân loại ngôn ngữ.


In [17]:
import re
import json
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import words as nltk_words

nltk.download("words", quiet=True)
ENGLISH_VOCAB = set(nltk_words.words())

# Bộ ký tự có dấu tiếng Việt — dùng cho bộ phân loại ngôn ngữ
VIETNAMESE_DIACRITIC_RE = re.compile(
    r"[àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệđìíỉĩịòóỏõọôồốổỗộơờớởỡợ"
    r"ùúủũụưừứửữựỳýỷỹỵ"
    r"ÀÁẢÃẠĂẰẮẲẴẶÂẦẤẨẪẬÈÉẺẼẸÊỀẾỂỄỆĐÌÍỈĨỊÒÓỎÕỌÔỒỐỔỖỘƠỜỚỞỠỢ"
    r"ÙÚỦŨỤƯỪỨỬỮỰỲÝỶỸỴ]"
)

# Từ tiếng Việt không dấu phổ biến dễ nhầm với tiếng Anh
VI_BARE_WORDS = {
    'cho', 'tai', 'nghe', 'thun', 'mi', 'treo', 'nam', 'kem', 'bao', 'da','loa', 
}

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


## 2. Tải dữ liệu


In [18]:
DATA_PATH = "/kaggle/input/datasets/hoangyendo/545151/data_cleaned.csv"
df = pd.read_csv(DATA_PATH)

assert "title_clean" in df.columns, "Thiếu cột title_clean"
assert "industry_group" in df.columns, "Thiếu cột industry_group"


## 3. Bộ phân loại đặc trưng

Ba đặc trưng phân tầng được tăng cường:

- **Ngôn ngữ** — tỉ lệ token tiếng Anh trong tổng word token không phải số.
- **Kiểu viết hoa/thường** — All Caps / Lower Case.
- **Nhóm độ dài** — Ngắn (≤12 từ), Trung bình (13–20), Dài (21–30), Rất dài (>30).

In [19]:
def classify_language(text: str) -> str:

    if not isinstance(text, str) or not text.strip():
        return "Unknown"

    tokens = [t.lower() for t in re.findall(r"\b\w+\b", text)]
    vi_count = en_count = 0

    for token in tokens:
        if re.match(r"^\d+\w*$", token) or re.match(r"^\w*\d+$", token):
            continue
        if VIETNAMESE_DIACRITIC_RE.search(token):
            vi_count += 1
        elif token in ENGLISH_VOCAB:
            en_count += 1
        elif token in VI_BARE_WORDS:
            vi_count += 1

    total = vi_count + en_count
    if total == 0:
        return "Chỉ số/Ký tự nhiễu"

    r_en = en_count / total
    if r_en >= 0.90:
        return "Thuần Anh"
    elif r_en <= 0.10:
        return "Thuần Việt"
    elif r_en > 0.50:
        return "Trộn lẫn (Thiên Anh)"
    else:
        return "Trộn lẫn (Thiên Việt)"


def classify_case(text: str) -> str:
    t = str(text).strip()
    if t.isupper():
        return "All Caps"
    elif t.islower():
        return "Lower Case"
    return "Mixed Case"


def classify_length(text: str) -> str:
    n = len(str(text).split())
    if n <= 12:
        return "Ngắn"
    elif n <= 20:
        return "Trung bình"
    elif n <= 30:
        return "Dài"
    return "Rất dài"


## 4. Phân bổ hạn ngạch ngành và bổ chính tầng thiểu số

Hàm `build_quota_pool` thực hiện ba thao tác theo thứ tự:

1. **Phân bổ hạn ngạch ban đầu** — lấy mẫu ngẫu nhiên theo tỉ lệ từ corpus đầu vào.
2. **Bổ chính tầng thiểu số** — đảm bảo ba tầng hiếm (Thuần Anh, All Caps, Lower Case) có tối thiểu 20 mẫu bằng cách bổ sung từ buffer; nếu buffer cạn, vét từ corpus sạch còn lại.
3. **Top-up về mốc 3.300** — phân bổ phần dư 60% cho Electronics & Tech và 40% cho Beauty & Health.

Hàm trả về `(pool_df, buffer_df)` — hai tập không trùng nhau.


In [20]:
# Hạn ngạch ngành ban đầu (tổng 3.000)
INDUSTRY_QUOTAS = {
    "home & living":         907,
    "electronics & tech":    837,
    "fashion & accessories": 789,
    "beauty & health":       467,
}

# Tầng thiểu số cần đảm bảo tối thiểu 20 mẫu
STRICT_MINORITY_TARGETS = {"Thuần Anh", "All Caps", "Lower Case"}
MINORITY_FLOOR = 20

# Mục tiêu tổng pool sau top-up
POOL_TARGET = 3300


def build_quota_pool(df_source: pd.DataFrame, random_seed: int = RANDOM_SEED):
    """
    Xây dựng annotation pool 3.300 tiêu đề từ corpus sạch.
    Trả về (pool_df, buffer_df).
    """
    rng = np.random.default_rng(random_seed)
    df = df_source.copy()
    df["_uid"] = df.index

    # Gán đặc trưng phân tầng
    df["_lang"] = df["title_clean"].apply(classify_language)
    df["_case"] = df["title_clean"].apply(classify_case)
    df["_len"]  = df["title_clean"].apply(classify_length)

    # ── Bước 1: Phân bổ hạn ngạch ban đầu ──
    pool_rows, buffer_rows = [], []
    for industry, quota in INDUSTRY_QUOTAS.items():
        ind_df = df[df["industry_group"] == industry].sample(frac=1, random_state=random_seed)
        pool_rows.append(ind_df.iloc[:quota])
        buffer_rows.append(ind_df.iloc[quota:])

    pool   = pd.concat(pool_rows,   ignore_index=True)
    buffer = pd.concat(buffer_rows, ignore_index=True)
    selected_uids = set(pool["_uid"])

    # ── Bước 2: Bổ chính tầng thiểu số ──
    feat_map = {"Thuần Anh": "_lang", "All Caps": "_case", "Lower Case": "_case"}
    for target_val, feat_col in feat_map.items():
        current = int((pool[feat_col] == target_val).sum())
        if current >= MINORITY_FLOOR:
            continue
        needed = MINORITY_FLOOR - current

        # Lấy từ buffer trước
        buf_match = buffer[buffer[feat_col] == target_val]
        take_n = min(needed, len(buf_match))
        if take_n > 0:
            chosen = buf_match.iloc[:take_n]
            pool   = pd.concat([pool, chosen], ignore_index=True)
            buffer = buffer[~buffer["_uid"].isin(chosen["_uid"])].reset_index(drop=True)
            selected_uids.update(chosen["_uid"])
            needed -= take_n

        # Vét từ corpus chưa chọn nếu vẫn thiếu
        if needed > 0:
            raw_remain = df[~df["_uid"].isin(selected_uids) & (df[feat_col] == target_val)]
            force_n = min(needed, len(raw_remain))
            if force_n > 0:
                chosen = raw_remain.iloc[:force_n]
                pool   = pd.concat([pool, chosen], ignore_index=True)
                buffer = buffer[~buffer["_uid"].isin(chosen["_uid"])].reset_index(drop=True)
                selected_uids.update(chosen["_uid"])

    # ── Bước 3: Top-up về mốc POOL_TARGET ──
    leftover = POOL_TARGET - len(pool)
    if leftover > 0:
        elec_share   = int(leftover * 0.60)
        beauty_share = leftover - elec_share
        for industry, share in [("electronics & tech", elec_share), ("beauty & health", beauty_share)]:
            candidates = buffer[buffer["industry_group"] == industry]
            take_n = min(share, len(candidates))
            if take_n > 0:
                chosen = candidates.iloc[:take_n]
                pool   = pd.concat([pool, chosen], ignore_index=True)
                buffer = buffer[~buffer["_uid"].isin(chosen["_uid"])].reset_index(drop=True)
                selected_uids.update(chosen["_uid"])

        # Vét buffer bất kỳ nếu vẫn chưa đủ
        still_need = POOL_TARGET - len(pool)
        if still_need > 0 and len(buffer) > 0:
            chosen = buffer.iloc[:still_need]
            pool   = pd.concat([pool, chosen], ignore_index=True)
            selected_uids.update(chosen["_uid"])

    # Dọn cột nội bộ
    _drop = ["_uid", "_lang", "_case", "_len"]
    keep  = [c for c in pool.columns if c not in _drop]
    pool_out   = pool[keep].reset_index(drop=True)
    buffer_out = df[~df["_uid"].isin(selected_uids)].drop(columns=_drop, errors="ignore").reset_index(drop=True)

    return pool_out, buffer_out


working_set, _ = build_quota_pool(df)


## 5. Lấy mẫu phân tầng 

Hàm `strict_stratified_sample` được gọi tuần tự trên phần còn lại sau mỗi lần lấy mẫu.  
Mỗi lần gọi trả về `(sampled_df, remaining_df)` — hai tập không trùng nhau, được đảm bảo qua `_uid`.

Thứ tự phụ thuộc giữa các lần gọi là bắt buộc: Pilot R1 → R2 → R3 → IAA → LLM Main.


In [21]:
def strict_stratified_sample(
    df_source: pd.DataFrame,
    target_n: int,
    random_seed: int = RANDOM_SEED,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Lấy target_n mẫu từ df_source theo phân tầng ngành,
    với bảo đảm tối thiểu 20 mẫu cho các tầng thiểu số.
    Trả về (sampled_df, remaining_df).
    """
    df = df_source.copy()
    df["_uid"] = df.index

    df["_lang"] = df["title_clean"].apply(classify_language)
    df["_case"] = df["title_clean"].apply(classify_case)
    df["_len"]  = df["title_clean"].apply(classify_length)

    # Hạn ngạch theo tỉ lệ từ INDUSTRY_QUOTAS gốc
    base_ratio = (target_n * 0.90) / sum(INDUSTRY_QUOTAS.values())
    quotas = {k: int(v * base_ratio) for k, v in INDUSTRY_QUOTAS.items()}

    pool_rows, buffer_rows = [], []
    for industry, quota in quotas.items():
        ind_df = df[df["industry_group"] == industry].sample(frac=1, random_state=random_seed)
        pool_rows.append(ind_df.iloc[:quota])
        buffer_rows.append(ind_df.iloc[quota:])

    pool   = pd.concat(pool_rows,   ignore_index=True)
    buffer = pd.concat(buffer_rows, ignore_index=True)
    selected_uids = set(pool["_uid"])

    # Bổ chính tầng thiểu số
    feat_map = {"Thuần Anh": "_lang", "All Caps": "_case", "Lower Case": "_case"}
    for target_val, feat_col in feat_map.items():
        current = int((pool[feat_col] == target_val).sum())
        if current >= MINORITY_FLOOR:
            continue
        needed = MINORITY_FLOOR - current

        buf_match = buffer[buffer[feat_col] == target_val]
        take_n = min(needed, len(buf_match))
        if take_n > 0:
            chosen = buf_match.iloc[:take_n]
            pool   = pd.concat([pool, chosen], ignore_index=True)
            buffer = buffer[~buffer["_uid"].isin(chosen["_uid"])].reset_index(drop=True)
            selected_uids.update(chosen["_uid"])
            needed -= take_n

        if needed > 0:
            raw_remain = df[~df["_uid"].isin(selected_uids) & (df[feat_col] == target_val)]
            force_n = min(needed, len(raw_remain))
            if force_n > 0:
                chosen = raw_remain.iloc[:force_n]
                pool   = pd.concat([pool, chosen], ignore_index=True)
                buffer = buffer[~buffer["_uid"].isin(chosen["_uid"])].reset_index(drop=True)
                selected_uids.update(chosen["_uid"])

    # Top-up đến target_n
    leftover = target_n - len(pool)
    if leftover > 0:
        elec_share   = int(leftover * 0.60)
        beauty_share = leftover - elec_share
        for industry, share in [("electronics & tech", elec_share), ("beauty & health", beauty_share)]:
            candidates = buffer[buffer["industry_group"] == industry]
            take_n = min(share, len(candidates))
            if take_n > 0:
                chosen = candidates.iloc[:take_n]
                pool   = pd.concat([pool, chosen], ignore_index=True)
                buffer = buffer[~buffer["_uid"].isin(chosen["_uid"])].reset_index(drop=True)
                selected_uids.update(chosen["_uid"])

        still_need = target_n - len(pool)
        if still_need > 0 and len(buffer) > 0:
            chosen = buffer.iloc[:still_need]
            pool   = pd.concat([pool, chosen], ignore_index=True)
            selected_uids.update(chosen["_uid"])

    elif leftover < 0:
        pool = pool.iloc[:target_n]
        selected_uids = set(pool["_uid"])

    _drop = ["_uid", "_lang", "_case", "_len"]
    keep  = [c for c in pool.columns if c not in _drop]
    sampled   = pool[keep].reset_index(drop=True)
    remaining = df[~df["_uid"].isin(selected_uids)].drop(columns=_drop, errors="ignore").reset_index(drop=True)

    return sampled, remaining

## 6. Thực thi lấy mẫu tuần tự

Năm lần gọi tuần tự, mỗi lần nhận phần còn lại của lần trước.  
Thứ tự: Pilot R1 (100) → Pilot R2 (100) → Pilot R3 (100) → IAA Cross-check (300) → LLM Main (2.700).


In [22]:
pilot_r1, remain_1 = strict_stratified_sample(working_set, target_n=100,  random_seed=RANDOM_SEED)
pilot_r2, remain_2 = strict_stratified_sample(remain_1,    target_n=100,  random_seed=RANDOM_SEED)
pilot_r3, remain_3 = strict_stratified_sample(remain_2,    target_n=100,  random_seed=RANDOM_SEED)
iaa_set,  remain_4 = strict_stratified_sample(remain_3,    target_n=300,  random_seed=RANDOM_SEED)
llm_main, _        = strict_stratified_sample(remain_4,    target_n=2700, random_seed=RANDOM_SEED)

subsets = {
    "pilot_r1": pilot_r1,
    "pilot_r2": pilot_r2,
    "pilot_r3": pilot_r3,
    "iaa_set":  iaa_set,
    "llm_main": llm_main,
}

## 7. Kiểm tra overlap trước khi xuất

Kiểm tra mọi cặp tập con không có tiêu đề trùng nhau.  
Nếu phát hiện overlap, pipeline dừng và không xuất file — tránh rò rỉ dữ liệu giữa các tập.


In [23]:
ID_COL = "title_clean"

sets_by_uid = {name: set(df[ID_COL]) for name, df in subsets.items()}
pairs = [
    ("pilot_r1", "pilot_r2"),
    ("pilot_r1", "pilot_r3"),
    ("pilot_r2", "pilot_r3"),
    ("pilot_r3", "iaa_set"),
    ("pilot_r3", "llm_main"),
    ("iaa_set",  "llm_main"),
]

overlap_found = False
for a, b in pairs:
    n_overlap = len(sets_by_uid[a] & sets_by_uid[b])
    if n_overlap > 0:
        print(f"OVERLAP: {a} & {b} — {n_overlap} tiêu đề trùng")
        overlap_found = True

if overlap_found:
    raise RuntimeError("Phát hiện overlap — dừng pipeline, không xuất file.")

## 8. Xuất file

Xuất năm tập con ra CSV. Mỗi file tương ứng với một giai đoạn trong quy trình gán nhãn.


In [24]:
OUTPUT_FILES = {
    "pilot_r1": "round1_pilot_100.csv",
    "pilot_r2": "round2_pilot_100.csv",
    "pilot_r3": "round3_pilot_100.csv",
    "iaa_set":  "campaign_iaa_cross_300.csv",
    "llm_main": "campaign_llm_main_2700.csv",
}

for name, df_out in subsets.items():
    path = OUTPUT_FILES[name]
    df_out.to_csv(path, index=False)